# Week 11 — Practical Lab: Clustering and Anomaly Detection

### Learning outcomes
By the end, you should be able to (1) explain the K-means assignment and centroid-update steps, (2) measure distortion and assess initialization, (3) compress images using color quantization, (4) choose K based on downstream quality and size, (5) fit an independent-feature Gaussian anomaly detector, (6) choose a threshold without using the test labels, and (7) diagnose false alarms and missed anomalies.

### Student instructions
Run the notebook from top to bottom. **Complete every numbered TODO** and write short interpretations where requested. A TODO cell may contain a working baseline; your task is to extend or replace the indicated part. Do not use test labels for model or threshold selection. Random seeds are fixed for reproducibility.

**Files:** `week11_compression_image.png`, `week11_anomaly_train.csv`, `week11_anomaly_validation.csv`, `week11_anomaly_test.csv`. Put them in the same folder as this notebook. The notebook also provides a fallback image if the PNG is missing.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `Pillow` (no TensorFlow required: these classical ML algorithms are implemented with NumPy/scikit-learn).

In [ ]:
# Setup — run this cell first
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from sklearn.cluster import KMeans
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

SEED = 2026
rng = np.random.default_rng(SEED)
DATA_DIR = Path('.')
plt.rcParams['figure.figsize'] = (8, 4.5)
print('Ready. NumPy:', np.__version__)

---
# Part A — K-means clustering for image compression

An RGB image has shape `(height, width, 3)`. Each pixel is a three-dimensional vector `[R,G,B]`. K-means finds **K representative colors**; each original pixel is replaced by its nearest representative. The output is a *lossy* approximation, not a pixel-perfect reconstruction.

K-means minimizes the within-cluster sum of squared distances:

$$J = \frac{1}{m}\sum_{i=1}^{m}\left\|x^{(i)}-\mu_{c^{(i)}}\right\|_2^2,$$

where $m$ is the number of pixels, $c^{(i)}\in\{1,\dots,K\}$ is the cluster assignment for pixel $i$, and $\mu_k\in\mathbb{R}^3$ is centroid $k$.

In [ ]:
# Load the supplied image; fallback is generated locally if the PNG is missing.
image_path = DATA_DIR / 'week11_compression_image.png'
if image_path.exists():
    image = np.asarray(Image.open(image_path).convert('RGB'))
else:
    yy, xx = np.mgrid[:180, :260]
    image = np.stack([70+140*xx/260, 60+130*yy/180, 160+60*np.sin(xx/25)], axis=-1)
    image = np.uint8(np.clip(image, 0, 255))
print('Image shape:', image.shape)
plt.imshow(image); plt.title('Original RGB image'); plt.axis('off'); plt.show()

### TODO 1 — Inspect image features
1. Print the image's height, width, channel count, and total number of pixels.
2. Reshape it to an array of shape `(m, 3)` and convert values to floating-point in `[0,1]`.
3. Print the first five RGB vectors and explain why pixels are *unlabeled* examples in this task.
4. Verify with an `assert` that the number of rows equals `height × width`.

In [ ]:
# TODO 1: replace/extend the starter code
height, width, channels = image.shape
pixels = image.reshape(-1, 3).astype(np.float64) / 255.0
print('Dimensions:', height, width, channels, '| Pixel count:', len(pixels))
print('First five RGB vectors:', pixels[:5])
assert pixels.shape == (height * width, 3)
# TODO: write a brief explanation in the Markdown cell below.

**TODO 1 explanation:** _Write your answer here._

### TODO 2 — Implement the K-means assignment step
1. Implement `assign_clusters(X, centroids)` using **squared Euclidean distance**.
2. Return a vector of length `m` with integer cluster indices `0` through `K−1`.
3. Do not use `sklearn` inside your implementation.
4. Verify the supplied two-point sanity check and explain why taking the square root is unnecessary.

In [ ]:
def assign_clusters(X, centroids):
    # TODO 2: calculate squared distances of shape (n_samples, K).
    # Hint: broadcasting: X[:, None, :] - centroids[None, :, :]
    squared_distances = np.sum((X[:, None, :] - centroids[None, :, :])**2, axis=2)
    return np.argmin(squared_distances, axis=1)

sample_X = np.array([[0., 0.], [9., 9.]])
sample_centroids = np.array([[1., 1.], [8., 8.]])
assert np.array_equal(assign_clusters(sample_X, sample_centroids), [0, 1])
print('Assignment check passed.')

### TODO 3 — Implement the centroid-update step
1. For each cluster, compute the arithmetic mean of the points assigned to it.
2. Handle an **empty cluster** explicitly: retain its previous centroid (or explain and implement another defensible strategy).
3. Check that the output has shape `(K, n_features)`.
4. Explain why moving a centroid to its assigned points' mean does not increase the squared-distance objective when assignments are fixed.

In [ ]:
def update_centroids(X, labels, old_centroids):
    # TODO 3: implement the update loop, including empty-cluster handling.
    new_centroids = old_centroids.copy()
    for k in range(len(old_centroids)):
        members = X[labels == k]
        if len(members) > 0:
            new_centroids[k] = members.mean(axis=0)
    return new_centroids

assert np.allclose(update_centroids(sample_X, np.array([0, 1]), sample_centroids), sample_X)
print('Centroid update check passed.')

### TODO 4 — Build one complete K-means run
1. Initialize K **distinct training examples** as centroids (sample without replacement).
2. Alternate assignment and centroid updates until labels stop changing or `max_iter` is reached.
3. Record the distortion after each full iteration, using the updated centroids and their nearest assignments.
4. Plot the objective across iterations and explain whether it is non-increasing (allowing small floating-point differences).
5. Try `K=6` on a random subset of 4,000 pixels for speed.

In [ ]:
def distortion(X, centroids):
    labels = assign_clusters(X, centroids)
    return np.mean(np.sum((X - centroids[labels])**2, axis=1))

def kmeans_from_scratch(X, K, seed=0, max_iter=30):
    # TODO 4: inspect and annotate the initialization/iteration/convergence logic.
    local_rng = np.random.default_rng(seed)
    centroids = X[local_rng.choice(len(X), K, replace=False)].copy()
    history = [distortion(X, centroids)]
    previous_labels = None
    for iteration in range(max_iter):
        labels = assign_clusters(X, centroids)
        centroids = update_centroids(X, labels, centroids)
        history.append(distortion(X, centroids))
        if previous_labels is not None and np.array_equal(labels, previous_labels):
            break
        previous_labels = labels.copy()
    return centroids, np.array(history)

subset = pixels[rng.choice(len(pixels), 4000, replace=False)]
centers_scratch, objective_history = kmeans_from_scratch(subset, K=6, seed=SEED)
plt.plot(np.arange(len(objective_history)), objective_history, marker='o')
plt.xlabel('Iteration'); plt.ylabel('Mean squared color distortion'); plt.title('K-means objective'); plt.show()
print('Initial / final distortion:', objective_history[0], objective_history[-1])

### TODO 5 — Compare random initializations
1. Run your implementation with **at least 10 different seeds** using the same `K=6` and pixel subset.
2. Record each run's final distortion, then report the minimum, median, and maximum.
3. Plot final distortion against seed.
4. Explain why selecting the run with the **lowest distortion** is preferable to running K-means once, and distinguish *number of restarts* from *number of clusters*.

In [ ]:
# TODO 5: change range(10) to a larger set if desired.
seeds = range(10)
final_costs = []
for seed in seeds:
    _, hist = kmeans_from_scratch(subset, K=6, seed=seed)
    final_costs.append(hist[-1])
plt.plot(list(seeds), final_costs, 'o-'); plt.xlabel('Random seed'); plt.ylabel('Final distortion'); plt.show()
print('Min / median / max:', np.min(final_costs), np.median(final_costs), np.max(final_costs))

### TODO 6 — Compress the image with scikit-learn
1. Fit `KMeans(n_clusters=K, random_state=SEED, n_init=10)` to RGB pixel vectors, for `K=8`.
2. Replace each pixel with the RGB centroid corresponding to its cluster label.
3. Reshape the compressed pixels back to `(height, width, 3)` and display original and compressed images side by side.
4. Report mean squared error (MSE) between original and reconstructed RGB arrays.
5. Explain why the compressed image has no more than K unique RGB colors.

In [ ]:
# TODO 6: experiment with K and inspect the quantized image.
def compress_image(rgb_pixels, image_shape, K, seed=SEED):
    model = KMeans(n_clusters=K, random_state=seed, n_init=10)
    labels = model.fit_predict(rgb_pixels)
    reconstructed = model.cluster_centers_[labels].reshape(image_shape)
    return np.clip(reconstructed, 0, 1), model

compressed8, model8 = compress_image(pixels, image.shape, K=8)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(image); axes[0].set_title('Original')
axes[1].imshow(compressed8); axes[1].set_title('K-means: K=8')
for ax in axes: ax.axis('off')
plt.show()
print('RGB MSE:', np.mean((pixels - compressed8.reshape(-1, 3))**2))

### TODO 7 — Choose K: elbow method and downstream quality
1. Evaluate at least `K ∈ {2, 4, 8, 16, 32}` using a **fixed pixel subset** for efficient model fitting.
2. Plot training distortion against K (elbow curve).
3. Reconstruct the **full image** with each fitted model and show the results in a row.
4. Compute reconstruction MSE for the full image.
5. Select a K that balances visual quality and the number of colors; explain why minimizing distortion alone favors larger K.

In [ ]:
# TODO 7: inspect each image and justify your chosen K.
Ks = [2, 4, 8, 16, 32]
models = {}
fit_costs, full_mses = [], []
fig, axes = plt.subplots(1, len(Ks), figsize=(16, 3.5))
for ax, K in zip(axes, Ks):
    model = KMeans(n_clusters=K, random_state=SEED, n_init=5).fit(subset)
    labels_all = model.predict(pixels)
    recon = np.clip(model.cluster_centers_[labels_all].reshape(image.shape), 0, 1)
    models[K] = model
    fit_costs.append(model.inertia_ / len(subset))
    full_mses.append(np.mean((pixels - recon.reshape(-1, 3))**2))
    ax.imshow(recon); ax.set_title(f'K={K}'); ax.axis('off')
plt.show()
plt.plot(Ks, fit_costs, 'o-', label='Fit distortion')
plt.plot(Ks, full_mses, 's--', label='Full-image MSE')
plt.xlabel('K (colors)'); plt.ylabel('Mean squared RGB error'); plt.legend(); plt.show()

### TODO 8 — Estimate compression and export
1. For an RGB image of `m` pixels, estimate raw storage as `24m` bits.
2. For a K-color palette, estimate storage as `24K + m × ceil(log2 K)` bits (palette + fixed-length pixel indices). **Ignore file headers and entropy coding**.
3. Calculate the approximate compression ratio for `K=8,16,32`.
4. Export a quantized PNG using `Image.fromarray`, and compare its actual file size with the original PNG. Explain why actual PNG sizes may not match the simplified estimate.
5. Write 3–4 sentences about whether the best K depends on downstream use (thumbnail, poster, scientific image).

In [ ]:
# TODO 8: extend the estimate to other K values.
m = len(pixels)
for K in [8, 16, 32]:
    bits_per_index = int(np.ceil(np.log2(K)))
    ratio = (24*m) / (24*K + m*bits_per_index)
    print(f'K={K:2d}: theoretical raw-to-palette ratio = {ratio:.2f}x')
output_png = DATA_DIR / 'week11_quantized_K8.png'
Image.fromarray(np.uint8(np.round(compressed8*255))).save(output_png)
if image_path.exists():
    print('Original PNG bytes:', image_path.stat().st_size)
print('Quantized PNG bytes:', output_png.stat().st_size)

---
# Part B — Gaussian anomaly detection on synthetic machine-monitoring data

The synthetic dataset simulates four measurements of a computer or server: CPU utilization, memory utilization, disk operations per second, and network traffic. Normal measurements are **correlated** and the last two are **right-skewed**. Anomalies include CPU spikes, memory leaks, disk surges, network surges, and combined high-load events.

**Data protocol:** Fit the density model on normal training examples. Use the labeled validation set to choose features, transformations, and the threshold. Use the labeled test set **once** for final reporting. The labels `0=normal`, `1=anomaly` are not features.

In [ ]:
# Load supplied CSVs (or generate matching synthetic files if they are absent).
feature_cols = ['cpu_percent', 'memory_percent', 'disk_ops_per_sec', 'network_mb_per_min']
train_path = DATA_DIR / 'week11_anomaly_train.csv'
val_path = DATA_DIR / 'week11_anomaly_validation.csv'
test_path = DATA_DIR / 'week11_anomaly_test.csv'
if not all(q.exists() for q in [train_path, val_path, test_path]):
    print('CSV files missing. Run the accompanying data-generation script or place supplied CSVs here.')
    raise FileNotFoundError('Missing Week 11 anomaly CSVs')
train = pd.read_csv(train_path)
val = pd.read_csv(val_path)
test = pd.read_csv(test_path)
print('Shapes:', train.shape, val.shape, test.shape)
print('Validation labels:', val['label'].value_counts().to_dict())
print('Test labels:', test['label'].value_counts().to_dict())

### TODO 9 — Inspect the dataset and identify anomaly features
1. Display the first five rows, column names, and descriptive statistics of each split.
2. Verify that all training labels are normal (`0`) and that validation/test splits contain both classes.
3. Draw a histogram for each feature on the training data.
4. Describe which features appear right-skewed and identify at least two abnormal machine behaviors the features could reveal.
5. Explain why `anomaly_type` and `label` must not be included in the density model's input.

In [ ]:
# TODO 9: inspect the summaries and distributions.
print(train.head())
print(train[feature_cols].describe().round(2))
assert (train['label'] == 0).all()
assert set(val['label'].unique()) == {0, 1}
assert set(test['label'].unique()) == {0, 1}
train[feature_cols].hist(bins=35, figsize=(11, 7))
plt.suptitle('Normal training feature distributions'); plt.tight_layout(); plt.show()

### TODO 10 — Estimate Gaussian parameters
1. Implement the per-feature maximum-likelihood mean `μ_j = (1/m) Σ_i x_j^(i)`.
2. Implement the maximum-likelihood variance `σ²_j = (1/m) Σ_i (x_j^(i) − μ_j)²` (**divide by m, not m−1**).
3. Verify against `np.mean(..., axis=0)` and `np.var(..., axis=0)`.
4. Explain what the mean and variance represent for CPU and memory measurements.

In [ ]:
def fit_independent_gaussian(X):
    # TODO 10: implement MLE mean and variance.
    mu = np.sum(X, axis=0) / len(X)
    var = np.sum((X - mu)**2, axis=0) / len(X)
    var = np.maximum(var, 1e-10)  # numerical safeguard
    return mu, var

X_train = train[feature_cols].to_numpy(dtype=float)
mu, var = fit_independent_gaussian(X_train)
assert np.allclose(mu, X_train.mean(axis=0))
assert np.allclose(var, X_train.var(axis=0))
print(pd.DataFrame({'feature':feature_cols, 'mean':mu, 'variance':var}))

### TODO 11 — Compute log density without numerical underflow
Under the **independent-feature approximation**, $p(x)=\prod_{j=1}^{n}\mathcal{N}(x_j;\mu_j,\sigma_j^2)$. Multiplying many tiny densities can underflow, so compute:

$$\log p(x)=-\frac12\sum_{j=1}^{n}\left[\log(2\pi\sigma_j^2)+\frac{(x_j-\mu_j)^2}{\sigma_j^2}\right].$$

1. Implement `gaussian_log_density(X, mu, var)` for a matrix of examples.
2. Return one log-density score per example.
3. Confirm that a point at the fitted mean scores higher than a point many standard deviations away.
4. Explain why `p(x)` is a **density**, not the probability of observing one exact continuous-valued vector.

In [ ]:
def gaussian_log_density(X, mu, var):
    # TODO 11: compute log density by summing across features.
    return -0.5 * np.sum(np.log(2*np.pi*var) + (X-mu)**2/var, axis=1)

logp_train = gaussian_log_density(X_train, mu, var)
assert gaussian_log_density(mu.reshape(1,-1), mu, var)[0] > gaussian_log_density((mu+8*np.sqrt(var)).reshape(1,-1), mu, var)[0]
print('Normal train log-density quantiles:', np.quantile(logp_train, [.01, .5, .99]))

### TODO 12 — Visualize density scores and the decision rule
1. Compute validation log-density scores using **training** mean/variance.
2. Plot separate histograms for validation normal and anomalous examples.
3. Identify whether the score distributions overlap and explain how overlap produces false positives and false negatives.
4. State the log-space decision rule: **anomaly if `log p(x) < log ε`**. Explain why lower scores indicate less typical observations under the fitted model.

In [ ]:
# TODO 12: add your observations below the plot.
X_val = val[feature_cols].to_numpy(dtype=float)
y_val = val['label'].to_numpy(dtype=int)
logp_val = gaussian_log_density(X_val, mu, var)
plt.hist(logp_val[y_val==0], bins=35, alpha=.65, label='Normal')
plt.hist(logp_val[y_val==1], bins=35, alpha=.65, label='Anomaly')
plt.xlabel('Log density (lower = less typical)'); plt.ylabel('Count'); plt.legend(); plt.show()

### TODO 13 — Tune the anomaly threshold on validation data
1. Consider a grid of candidate thresholds based on validation score quantiles.
2. Predict an anomaly when `logp < threshold`.
3. Compute precision, recall, and F1 for every candidate.
4. Choose the threshold maximizing **validation F1**; print the best threshold and metrics.
5. Explain what happens to precision and recall when the threshold is raised, and why this choice must not use test labels.

*Note:* In this notebook the threshold variable is `log_epsilon` because scores are log densities.

In [ ]:
# TODO 13: replace the quantile grid with a finer grid and inspect changes.
thresholds = np.unique(np.quantile(logp_val, np.linspace(0.001, 0.999, 300)))
rows = []
for t in thresholds:
    pred = (logp_val < t).astype(int)
    rows.append((t, precision_score(y_val,pred,zero_division=0),
                 recall_score(y_val,pred,zero_division=0), f1_score(y_val,pred,zero_division=0)))
results = pd.DataFrame(rows, columns=['log_epsilon','precision','recall','f1'])
best_row = results.loc[results['f1'].idxmax()]
log_epsilon = float(best_row['log_epsilon'])
print('Best validation threshold and metrics:', best_row)
plt.plot(results['log_epsilon'], results['precision'], label='Precision')
plt.plot(results['log_epsilon'], results['recall'], label='Recall')
plt.plot(results['log_epsilon'], results['f1'], label='F1')
plt.axvline(log_epsilon, linestyle='--', label='Chosen threshold')
plt.xlabel('Log-density threshold'); plt.legend(); plt.show()

### TODO 14 — Diagnose errors with a confusion matrix
1. Predict validation labels with the chosen threshold.
2. Plot the confusion matrix with classes `Normal` and `Anomaly`.
3. Count false positives and false negatives and explain each in the computer-monitoring context.
4. Print at least three incorrectly classified examples (if available) with their feature values, true label, predicted label, and log density.
5. Explain whether missed anomalies have scores similar to normal examples.

In [ ]:
# TODO 14: inspect false alarms and missed anomalies.
val_pred = (logp_val < log_epsilon).astype(int)
cm = confusion_matrix(y_val, val_pred, labels=[0,1])
ConfusionMatrixDisplay(cm, display_labels=['Normal','Anomaly']).plot(cmap='Blues')
plt.show()
errors = val.loc[val_pred != y_val, feature_cols+['label','anomaly_type']].copy()
errors['predicted'] = val_pred[val_pred != y_val]
errors['log_density'] = logp_val[val_pred != y_val]
print('False positives:', cm[0,1], '| False negatives:', cm[1,0])
print(errors.head(6).to_string(index=False))

### TODO 15 — Transform skewed features
1. Apply `log1p` to disk operations and network traffic; leave CPU and memory on their original scales.
2. Fit a **new** independent Gaussian model on transformed training data.
3. Compute transformed validation log densities and tune a new threshold on validation F1.
4. Compare baseline and transformed-model precision, recall, F1, and confusion matrices.
5. Explain why transforming positive skewed features may improve a Gaussian fit, and why it does **not** guarantee improved anomaly detection.

**Important:** Compare models through **classification metrics**, not raw density magnitudes across different feature spaces (which are not directly comparable).

In [ ]:
# TODO 15: extend to other transformations and compare validation metrics.
def transform_features(X):
    Z = np.array(X, dtype=float, copy=True)
    Z[:, 2:] = np.log1p(np.maximum(Z[:, 2:], 0))
    return Z

X_train_t, X_val_t = transform_features(X_train), transform_features(X_val)
mu_t, var_t = fit_independent_gaussian(X_train_t)
logp_val_t = gaussian_log_density(X_val_t, mu_t, var_t)

def tune_threshold(scores, labels, n_grid=300):
    candidates = np.unique(np.quantile(scores, np.linspace(.001,.999,n_grid)))
    best = (-1., None, None)
    for threshold in candidates:
        pred = (scores < threshold).astype(int)
        f1 = f1_score(labels,pred,zero_division=0)
        if f1 > best[0]: best = (f1, float(threshold), pred)
    return best

f1_t, log_epsilon_t, val_pred_t = tune_threshold(logp_val_t, y_val)
for name, pred in [('Raw',val_pred), ('Log transformed',val_pred_t)]:
    print(name, 'precision=',round(precision_score(y_val,pred,zero_division=0),3),
          'recall=',round(recall_score(y_val,pred,zero_division=0),3),
          'F1=',round(f1_score(y_val,pred,zero_division=0),3))

### TODO 16 — Investigate feature dependence and error patterns
1. Plot the normal training data's feature correlation matrix.
2. Identify the pair of features with the strongest absolute off-diagonal correlation.
3. Explain why multiplying marginal Gaussian densities is an **approximation** when features are correlated.
4. Group validation errors by `anomaly_type` and count missed anomalies for each type.
5. Propose a new feature (e.g., CPU-to-memory ratio, or CPU–network interaction) that might help identify a missed anomaly. Explain how you would validate its usefulness.

In [ ]:
# TODO 16: interpret the correlation matrix and missed anomaly categories.
corr = train[feature_cols].corr()
fig, ax = plt.subplots(figsize=(6,5))
imshow = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(4),feature_cols,rotation=35,ha='right')
ax.set_yticks(range(4),feature_cols)
for i in range(4):
    for j in range(4): ax.text(j,i,f'{corr.iloc[i,j]:.2f}',ha='center',va='center')
fig.colorbar(imshow,ax=ax); plt.title('Normal-training feature correlations'); plt.tight_layout(); plt.show()
missed = val[(y_val==1) & (val_pred_t==0)]
print('Missed anomalies by type:',missed['anomaly_type'].value_counts())

### TODO 17 — Final held-out test evaluation
1. **Freeze** the selected transformed model and its threshold from TODO 15; do not refit or retune on test data.
2. Compute test log densities, predictions, precision, recall, F1, and confusion matrix.
3. Compare test and validation F1 and discuss possible sampling variation.
4. Explain the difference between a false alarm and a missed anomaly in an operational monitoring system.
5. State one limitation of evaluating on synthetic anomalies rather than real incident data.

In [ ]:
# TODO 17: report these values in your final lab write-up.
X_test = test[feature_cols].to_numpy(dtype=float)
y_test = test['label'].to_numpy(dtype=int)
logp_test_t = gaussian_log_density(transform_features(X_test), mu_t, var_t)
test_pred = (logp_test_t < log_epsilon_t).astype(int)
print('Held-out TEST precision:', precision_score(y_test,test_pred,zero_division=0))
print('Held-out TEST recall:', recall_score(y_test,test_pred,zero_division=0))
print('Held-out TEST F1:', f1_score(y_test,test_pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,test_pred,display_labels=['Normal','Anomaly'],cmap='Blues')
plt.title('Final test confusion matrix'); plt.show()

### TODO 18 — Extension: clustering versus anomaly detection
1. Fit K-means with `K=3` on **standardized normal training features** (after `log1p` transformation).
2. For each validation example, compute its squared distance to its nearest cluster center.
3. Treat large distances as anomaly scores and choose a validation threshold maximizing F1.
4. Compare its validation F1 with your Gaussian detector's validation F1.
5. Discuss why K-means clustering is **not itself an anomaly detector**, even though distance to a centroid can be used as an anomaly score.
6. Explain why the scaler and K-means model must be fitted on the training split only.

In [ ]:
# TODO 18: analyze whether nearest-centroid distance separates anomalies.
scaler = StandardScaler().fit(X_train_t)
Z_train = scaler.transform(X_train_t)
Z_val = scaler.transform(X_val_t)
km = KMeans(n_clusters=3,random_state=SEED,n_init=10).fit(Z_train)
dist2_val = np.min(np.sum((Z_val[:,None,:]-km.cluster_centers_[None,:,:])**2,axis=2),axis=1)
# A high distance indicates anomaly; negate scores to reuse the '< threshold' tuner.
km_f1, negative_threshold, km_pred = tune_threshold(-dist2_val, y_val)
print('K-means-distance validation F1:',km_f1)
print('Gaussian transformed validation F1:',f1_t)

---
# Lab submission checklist

Submit **(a)** the completed notebook with executed outputs and **(b)** a short 1–2 page interpretation (or Markdown answers in the notebook) addressing:

- Your chosen K for image compression, the resulting image quality, distortion, and approximate compression ratio.
- How multiple random initializations affected the K-means objective.
- Which anomaly features were skewed and whether `log1p` helped.
- How you selected the threshold using validation data and your final held-out test metrics.
- Two concrete error cases and a justified improvement proposal.
- A brief comparison of K-means distance and Gaussian density as anomaly scores.

**Academic integrity:** Complete the TODO reasoning yourself, retain your plots and computed results, and distinguish observations from hypotheses. The synthetic dataset is intended for teaching and does not establish real-world detector performance.

### Optional challenge
Implement a full-covariance multivariate Gaussian model and compare it with the independent-feature Gaussian model on validation data. Discuss covariance estimation stability and why log density is preferred numerically.